# LAB | BrightCart Daily Sales Pipeline: Build

**Week 7, Day 2**

BrightCart receives customer and order files every day. The pipeline must validate records, reject unsafe rows, join customers to orders, create derived values, preserve raw data, build a report, and record useful audit information.

Every exercise is independent and creates its own data. Each solution is commented to explain **why** each step exists, not only what it does.

| Part | Questions | Topic |
|---|---|---|
| 1 | Q1 to Q4 | Source data and validation |
| 2 | Q5 to Q8 | Rejecting, fixing and safe joins |
| 3 | Q9 to Q12 | Derived columns, curated output and raw preservation |
| 4 | Q13 to Q16 | Audit records and data quality tests |
| Optional | Q17 to Q20 | Full mini pipeline, dangerous joins, rejection reasons, audit design |

> Note: this lab writes the status as `complete`, while the lesson uses `COMPLETED`. Always compare against the exact value your data uses, or standardise it first.

In [1]:
# Shared imports for the whole notebook
import pandas as pd
import numpy as np

---
# Part 1: Source Data and Validation

## Q1. Validate a Customer Batch

**Rule:** `customer_id` and `customer_name` are required. `email` can be blank.

Create `valid_customers` and `invalid_customers`.

In [2]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "customer_name": ["Amelia Clarke", "Oliver Bennett", "Aisha Khan", "Harry Thompson"],
    "email": ["amelia@example.com", "", "aisha@example.com", None],
})

# A row is invalid when a REQUIRED field is missing.
# We only check the required columns. Email is optional, so it is NOT part of the rule.
invalid_mask = (
    customers["customer_id"].isna()
    | customers["customer_name"].isna()
)

# ~ means "NOT": valid rows are the ones that are not invalid.
# .copy() creates an independent DataFrame, so later changes do not affect `customers`.
valid_customers = customers[~invalid_mask].copy()
invalid_customers = customers[invalid_mask].copy()

print("Valid customers:")
print(valid_customers)
print("\nInvalid customers:")
print(invalid_customers)

Valid customers:
   customer_id   customer_name               email
0            1   Amelia Clarke  amelia@example.com
1            2  Oliver Bennett                    
2            3      Aisha Khan   aisha@example.com
3            4  Harry Thompson                 NaN

Invalid customers:
Empty DataFrame
Columns: [customer_id, customer_name, email]
Index: []


**What to notice:** Oliver (blank email) and Harry (`None` email) are still **valid**. A missing value is only a problem when the field is required for the business purpose. Dropping every row with any null (`dropna()`) would wrongly lose two good customers.

`invalid_customers` is **empty**, and that is a correct result: this batch has no problems. A good pipeline still produces the rejected output (even empty), so the audit can show `rejected_count = 0` instead of leaving people guessing.

## Q2. Validate an Order Batch

Reject rows when quantity is 0 or negative, or when the price is missing. Create valid and rejected DataFrames.

In [3]:
orders = pd.DataFrame({
    "order_id": [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 3, 4, 1],
    "quantity": [2, 1, 0, 3, 1],
    "unit_price_gbp": [25.00, 40.00, 12.00, None, 50.00],
    "status": ["complete", "complete", "complete", "complete", "cancelled"],
})

# Each rule is written as its own boolean Series, so the logic is easy to read and change.
invalid_quantity = orders["quantity"] <= 0
missing_price = orders["unit_price_gbp"].isna()

# A row is rejected if it breaks ANY rule (| means OR).
reject_mask = invalid_quantity | missing_price

valid_orders = orders[~reject_mask].copy()
rejected_orders = orders[reject_mask].copy()

print("Valid orders:")
print(valid_orders)
print("\nRejected orders:")
print(rejected_orders)

Valid orders:
   order_id  customer_id  quantity  unit_price_gbp     status
0       101            1         2            25.0   complete
1       102            2         1            40.0   complete
4       105            1         1            50.0  cancelled

Rejected orders:
   order_id  customer_id  quantity  unit_price_gbp    status
2       103            3         0            12.0  complete
3       104            4         3             NaN  complete


**What to notice:** order 105 is `cancelled` but it is still **valid**. Validation asks "is this row safe to use?". Whether it counts as revenue is a separate **reporting** decision, made later.

## Q3. Validate Three Daily Batches

A valid amount must be greater than 0. For each batch, print `source_count`, `valid_count` and `rejected_count`.

In [4]:
batch_1 = pd.DataFrame({"order_id": [201, 202], "amount_gbp": [45.50, 30.00]})
batch_2 = pd.DataFrame({"order_id": [203, 204], "amount_gbp": [0, 62.25]})
batch_3 = pd.DataFrame({"order_id": [205, 206], "amount_gbp": [-5.00, 20.00]})

# A dictionary lets us apply the SAME rule to every batch with one loop,
# instead of copying the same code three times.
batches = {"batch_1": batch_1, "batch_2": batch_2, "batch_3": batch_3}

for name, batch in batches.items():
    valid_mask = batch["amount_gbp"] > 0

    source_count = len(batch)
    valid_count = int(valid_mask.sum())        # True counts as 1, False as 0
    rejected_count = int((~valid_mask).sum())

    print(f"{name}: source_count={source_count}, valid_count={valid_count}, rejected_count={rejected_count}")

batch_1: source_count=2, valid_count=2, rejected_count=0
batch_2: source_count=2, valid_count=1, rejected_count=1
batch_3: source_count=2, valid_count=1, rejected_count=1


## Q4. MCQ: Validation

Why should row counts be checked during validation?

A. To confirm that rows were not silently lost or unexpectedly added
B. To prove every business value is correct automatically
C. To remove the need for business rules
D. To make all files the same size

**Answer: A.** Counts prove that every source row ended up somewhere (valid or rejected). They do **not** prove the values are correct (B), and they never replace business rules (C).

---
# Part 2: Rejecting, Fixing, and Safe Joins

## Q5. Reject or Fix?

For each issue, choose **fix**, **reject**, or **review**.

| Issue | Decision | Why |
|---|---|---|
| A. City contains extra spaces: `" London "` | **Fix** | Trimming spaces does not change the meaning. It is a safe, predictable correction. |
| B. Unit price is missing | **Reject** | We cannot calculate revenue, and inventing a price would create false numbers. |
| C. Email is blank but email is optional | **Keep (no action)** | The row already follows the rules. Nothing needs fixing or rejecting. |
| D. Quantity is `-3` | **Reject** | A negative quantity is not a valid sale. We do not know the correct value. Could also go to **review** if returns are possible in the business. |
| E. City is `"london"` but the standard is `"London"` | **Fix** | Standardising capitalisation is safe and keeps grouping consistent. |

**Rule of thumb:** fix when the correction is certain and does not change the meaning; reject when the correct value is unknown; review when a human needs to decide.

## Q6. Fix Safe Values and Reject Unsafe Values

Rules: trim spaces, convert city to title case, reject non positive quantity, reject missing price.

In [5]:
orders = pd.DataFrame({
    "order_id": [301, 302, 303, 304],
    "city": [" London ", "leeds", "Bristol", "Manchester"],
    "quantity": [2, 1, -1, 3],
    "unit_price_gbp": [25.00, 20.00, 15.00, None],
})

# Work on a copy, so the original data stays untouched.
clean = orders.copy()

# FIX first: safe corrections that do not change meaning.
# .str.strip() removes spaces at both ends, .str.title() gives "London", "Leeds"...
clean["city"] = clean["city"].str.strip().str.title()

# THEN REJECT: rows we cannot trust.
reject_mask = (clean["quantity"] <= 0) | (clean["unit_price_gbp"].isna())

valid_orders = clean[~reject_mask].copy()
rejected_orders = clean[reject_mask].copy()

print("Valid orders:")
print(valid_orders)
print("\nRejected orders:")
print(rejected_orders)

Valid orders:
   order_id    city  quantity  unit_price_gbp
0       301  London         2            25.0
1       302   Leeds         1            20.0

Rejected orders:
   order_id        city  quantity  unit_price_gbp
2       303     Bristol        -1            15.0
3       304  Manchester         3             NaN


## Q7. Perform a Safe Many to One Join

Join orders to customers and use pandas join validation to confirm that many orders can match one customer.

In [6]:
customers = pd.DataFrame({
    "customer_id": [1, 2, 3],
    "customer_name": ["Amelia Clarke", "Oliver Bennett", "Aisha Khan"],
    "city": ["London", "Manchester", "Leeds"],
})

orders = pd.DataFrame({
    "order_id": [401, 402, 403, 404],
    "customer_id": [1, 2, 1, 3],
    "amount_gbp": [45.50, 30.00, 62.25, 20.00],
})

# how="left"  keeps EVERY order, even if its customer is not found.
# validate="many_to_one" tells pandas what we expect:
#   many orders  ->  one customer.
# If the customers table had a repeated customer_id, pandas would raise an error
# instead of silently duplicating order rows.
merged = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one",
)

print(merged)

# Extra safety check: a left join must never change the number of orders.
print("\nOrders before:", len(orders), "| rows after join:", len(merged))

   order_id  customer_id  amount_gbp   customer_name        city
0       401            1       45.50   Amelia Clarke      London
1       402            2       30.00  Oliver Bennett  Manchester
2       403            1       62.25   Amelia Clarke      London
3       404            3       20.00      Aisha Khan       Leeds

Orders before: 4 | rows after join: 4


## Q8. MCQ: Safe Join

Why validate a many to one join?

A. To make the join result sorted
B. To remove every unmatched row automatically
C. To convert all columns to text
D. To check that the lookup side does not unexpectedly contain duplicate keys

**Answer: D.** If the customers table had two rows with the same `customer_id`, every matching order would be duplicated, and revenue would be overstated. Validation makes the join fail loudly instead.

---
# Part 3: Derived Columns, Curated Output, and Raw Preservation

## Q9. Create a Derived Revenue Column

Create `line_revenue_gbp = quantity × unit_price_gbp`.

In [7]:
orders = pd.DataFrame({
    "order_id": [501, 502, 503],
    "quantity": [2, 1, 4],
    "unit_price_gbp": [25.00, 40.00, 12.50],
})

# A derived column is calculated from other columns, it does not come from the source.
# pandas multiplies row by row (vectorised), no loop needed.
orders["line_revenue_gbp"] = orders["quantity"] * orders["unit_price_gbp"]

print(orders)

   order_id  quantity  unit_price_gbp  line_revenue_gbp
0       501         2            25.0              50.0
1       502         1            40.0              40.0
2       503         4            12.5              50.0


**In a real pipeline:** create derived columns only **after** validation. Multiplying by a missing price gives `NaN`, and a negative quantity gives negative revenue.

## Q10. Build a Curated Daily Sales Report

Create a report with `order_date`, `city` and `completed_revenue_gbp`.

In [8]:
orders = pd.DataFrame({
    "order_date": ["2026-08-01", "2026-08-01", "2026-08-01", "2026-08-02"],
    "city": ["London", "London", "Manchester", "Leeds"],
    "amount_gbp": [45.50, 62.25, 30.00, 20.00],
    "status": ["complete", "complete", "cancelled", "complete"],
})

# Step 1: apply the business rule, only completed orders count as revenue.
completed = orders[orders["status"] == "complete"]

# Step 2: aggregate to the level the business asked for (one row per date and city).
# as_index=False keeps order_date and city as normal columns.
daily_sales = (
    completed
    .groupby(["order_date", "city"], as_index=False)["amount_gbp"]
    .sum()
    .rename(columns={"amount_gbp": "completed_revenue_gbp"})
)

print(daily_sales)

   order_date    city  completed_revenue_gbp
0  2026-08-01  London                 107.75
1  2026-08-02   Leeds                  20.00


**What to notice:** Manchester disappears because its only order was cancelled. The **cleaned** data would still keep that order. The **curated** report is shaped for one business question.

## Q11. Preserve Raw Data

Create `raw_orders` and `clean_orders`. Trim spaces only in `clean_orders`, then prove that `raw_orders` still contains `" London "`.

In [9]:
source = pd.DataFrame({
    "order_id": [601, 602, 603],
    "city": [" London ", "Leeds", "Manchester"],
})

# raw_orders is our untouched record of what arrived.
raw_orders = source.copy()

# clean_orders is a SEPARATE copy. Without .copy(), both names could point
# to the same data, and cleaning one would change the other.
clean_orders = raw_orders.copy()
clean_orders["city"] = clean_orders["city"].str.strip()

print("Raw:", raw_orders["city"].tolist())
print("Clean:", clean_orders["city"].tolist())

# Proof: an assert stops the code with an error if the condition is False.
assert raw_orders.loc[0, "city"] == " London "
assert clean_orders.loc[0, "city"] == "London"
print("\nRaw data preserved")

Raw: [' London ', 'Leeds', 'Manchester']
Clean: ['London', 'Leeds', 'Manchester']

Raw data preserved


## Q12. MCQ: Derived Column

Which is a derived column?

A. `customer_id` copied directly from the source
B. `line_revenue_gbp = quantity × unit_price_gbp`
C. `order_id` received from the source
D. `city` received from the source

**Answer: B.** It is calculated by the pipeline. All other options come straight from the source.

---
# Part 4: Audit Records and Data Quality Tests

## Q13. Create an Audit Record

Create a dictionary for the run and check that `source_count == valid_count + rejected_count`.

In [10]:
audit = {
    "run_id": "run_20260801",
    "source_count": 100,
    "valid_count": 94,
    "rejected_count": 6,
    "status": "success",
}

# The reconciliation check: every source row must be either valid or rejected.
audit["reconciles"] = audit["source_count"] == audit["valid_count"] + audit["rejected_count"]

print(audit)

{'run_id': 'run_20260801', 'source_count': 100, 'valid_count': 94, 'rejected_count': 6, 'status': 'success', 'reconciles': True}


**Why a dictionary?** It has named fields, so it can be saved as JSON and read later by a person or another program.

## Q14. Run Data Quality Tests

Calculate duplicate order IDs, missing customer IDs and non positive amounts.

In [11]:
orders = pd.DataFrame({
    "order_id": [701, 702, 702, 704],
    "customer_id": [1, 2, 2, None],
    "amount_gbp": [45.50, 30.00, 30.00, -5.00],
})

# duplicated() marks the 2nd, 3rd... occurrence of a value as True (the 1st stays False).
duplicate_order_ids = int(orders["order_id"].duplicated().sum())

missing_customer_ids = int(orders["customer_id"].isna().sum())

non_positive_amounts = int((orders["amount_gbp"] <= 0).sum())

print("duplicate order IDs:", duplicate_order_ids)
print("missing customer IDs:", missing_customer_ids)
print("non positive amounts:", non_positive_amounts)

# Showing WHICH rows fail is more useful than just a number when investigating.
print("\nRows with a repeated order_id:")
print(orders[orders["order_id"].duplicated(keep=False)])

duplicate order IDs: 1
missing customer IDs: 1
non positive amounts: 1

Rows with a repeated order_id:
   order_id  customer_id  amount_gbp
1       702          2.0        30.0
2       702          2.0        30.0


## Q15. Reconcile Three Batch Audits

Print whether each batch reconciles.

In [12]:
audit_rows = [
    {"batch": "batch_1", "source": 100, "valid": 97, "rejected": 3},
    {"batch": "batch_2", "source": 120, "valid": 118, "rejected": 2},
    {"batch": "batch_3", "source": 90, "valid": 88, "rejected": 1},
]

for row in audit_rows:
    accounted_for = row["valid"] + row["rejected"]
    reconciles = row["source"] == accounted_for

    if reconciles:
        print(f"{row['batch']}: reconciles ({row['source']} = {row['valid']} + {row['rejected']})")
    else:
        missing = row["source"] - accounted_for
        print(f"{row['batch']}: DOES NOT reconcile, {missing} row(s) unaccounted for")

batch_1: reconciles (100 = 97 + 3)
batch_2: reconciles (120 = 118 + 2)
batch_3: DOES NOT reconcile, 1 row(s) unaccounted for


**What to notice:** `batch_3` does not reconcile: 90 rows arrived but only 89 are accounted for. One row was **silently lost**. This is exactly the kind of problem the audit exists to catch. The run should not be reported as successful until it is investigated.

## Q16. MCQ: Audit Record

Which audit value is most useful for checking whether records were lost during validation?

A. Only the city name
B. Only the file name
C. Source count, valid count, and rejected count
D. Only the Python version

**Answer: C.** Only these three together let you check `source = valid + rejected`.

---
# Optional Practice

## Q17. Build a Complete Independent Daily Sales Pipeline

Create `raw_orders`, `valid_orders`, `rejected_orders`, `daily_sales` and `audit`.

Rules: preserve raw rows; reject quantity <= 0; reject missing price; trim city; calculate `line_revenue_gbp`; include only completed orders in daily sales; reconcile source, valid and rejected counts.

This exercise is the **whole project in miniature**. Each step below becomes one function of the BrightCart pipeline later.

In [13]:
batch_1 = pd.DataFrame({
    "order_id": [801, 802],
    "customer_id": [1, 2],
    "city": ["London", "Manchester"],
    "quantity": [2, 1],
    "unit_price_gbp": [25.00, 30.00],
    "status": ["complete", "complete"],
})
batch_2 = pd.DataFrame({
    "order_id": [803, 804],
    "customer_id": [1, 3],
    "city": [" London ", "Leeds"],
    "quantity": [1, 0],
    "unit_price_gbp": [40.00, 20.00],
    "status": ["complete", "complete"],
})
batch_3 = pd.DataFrame({
    "order_id": [805, 806],
    "customer_id": [4, 2],
    "city": ["Bristol", "Manchester"],
    "quantity": [3, 1],
    "unit_price_gbp": [None, 50.00],
    "status": ["complete", "cancelled"],
})

# 1. EXTRACT: combine the batches. raw_orders is never modified after this line.
raw_orders = pd.concat([batch_1, batch_2, batch_3], ignore_index=True)

# 2. CLEAN: safe fixes on a separate copy.
clean = raw_orders.copy()
clean["city"] = clean["city"].str.strip()

# 3. VALIDATE: split into valid and rejected. Nothing is dropped silently.
reject_mask = (clean["quantity"] <= 0) | (clean["unit_price_gbp"].isna())
valid_orders = clean[~reject_mask].copy()
rejected_orders = clean[reject_mask].copy()

# 4. DERIVE: revenue only on validated rows.
valid_orders["line_revenue_gbp"] = valid_orders["quantity"] * valid_orders["unit_price_gbp"]

# 5. CURATE: business report, completed orders only.
daily_sales = (
    valid_orders[valid_orders["status"] == "complete"]
    .groupby("city", as_index=False)["line_revenue_gbp"]
    .sum()
    .rename(columns={"line_revenue_gbp": "completed_revenue_gbp"})
)

# 6. AUDIT: evidence that the run can be trusted.
audit = {
    "source_count": len(raw_orders),
    "valid_count": len(valid_orders),
    "rejected_count": len(rejected_orders),
    "completed_revenue_gbp": float(daily_sales["completed_revenue_gbp"].sum()),
}
audit["reconciles"] = audit["source_count"] == audit["valid_count"] + audit["rejected_count"]

print("Raw:"); print(raw_orders)
print("\nValid:"); print(valid_orders)
print("\nRejected:"); print(rejected_orders)
print("\nDaily sales:"); print(daily_sales)
print("\nAudit:"); print(audit)

Raw:
   order_id  customer_id        city  quantity  unit_price_gbp     status
0       801            1      London         2            25.0   complete
1       802            2  Manchester         1            30.0   complete
2       803            1     London          1            40.0   complete
3       804            3       Leeds         0            20.0   complete
4       805            4     Bristol         3             NaN   complete
5       806            2  Manchester         1            50.0  cancelled

Valid:
   order_id  customer_id        city  quantity  unit_price_gbp     status  \
0       801            1      London         2            25.0   complete   
1       802            2  Manchester         1            30.0   complete   
2       803            1      London         1            40.0   complete   
5       806            2  Manchester         1            50.0  cancelled   

   line_revenue_gbp  
0              50.0  
1              30.0  
2              40

**What to notice:**

* Order 803 had the city `" London "`. Without the trim, the report would show **two London rows** ("London" with 50 and " London " with 40). After the trim, London = 50 + 40 = **90**.
* Order 806 is valid but cancelled: it is in `valid_orders` but not in `daily_sales`.
* `6 = 4 + 2`, so the run reconciles.

## Q18. Detect a Dangerous Join

Try a many to one join validation and explain why it should fail.

In [14]:
customers = pd.DataFrame({
    "customer_id": [1, 1, 2],
    "customer_name": ["Amelia Clarke", "Amelia Clarke Duplicate", "Oliver Bennett"],
})
orders = pd.DataFrame({
    "order_id": [901, 902],
    "customer_id": [1, 2],
    "amount_gbp": [45.50, 30.00],
})

# try / except lets us catch the error and explain it, instead of crashing the notebook.
try:
    orders.merge(customers, on="customer_id", how="left", validate="many_to_one")
    print("Join succeeded")
except pd.errors.MergeError as error:
    print("Join blocked:", error)

# What would happen WITHOUT validation: order 901 is duplicated.
unsafe = orders.merge(customers, on="customer_id", how="left")
print("\nWithout validation:", len(orders), "orders became", len(unsafe), "rows")
print(unsafe)
print("\nRevenue before:", orders["amount_gbp"].sum(), "| after unsafe join:", unsafe["amount_gbp"].sum())

Join blocked: Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
  customer_id
           1 ...

Without validation: 2 orders became 3 rows
   order_id  customer_id  amount_gbp            customer_name
0       901            1        45.5            Amelia Clarke
1       901            1        45.5  Amelia Clarke Duplicate
2       902            2        30.0           Oliver Bennett

Revenue before: 75.5 | after unsafe join: 121.0


**Explanation:** `customer_id = 1` appears twice in the customers table, so the "one" side is not unique. Without validation, order 901 matches both rows, appears twice, and revenue grows from 75.50 to 121.00 without anyone noticing. The validation turns a silent error into a loud one.

## Q19. Write a Rejection Reason

Create a `rejection_reason` column with the values `invalid_quantity`, `missing_price` or `valid`.

In [15]:
orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003],
    "quantity": [0, 2, 3],
    "unit_price_gbp": [10.00, None, 20.00],
})

# np.select checks the conditions IN ORDER and uses the first one that is True.
# If a row broke both rules, it would get the first reason in the list.
conditions = [
    orders["quantity"] <= 0,
    orders["unit_price_gbp"].isna(),
]
reasons = ["invalid_quantity", "missing_price"]

orders["rejection_reason"] = np.select(conditions, reasons, default="valid")

print(orders)

   order_id  quantity  unit_price_gbp  rejection_reason
0      1001         0            10.0  invalid_quantity
1      1002         2             NaN     missing_price
2      1003         3            20.0             valid


**Why it matters:** a rejected file without reasons forces someone to investigate each row by hand. With a reason column, the operations team can count problems by type (`value_counts()`) and fix the source.

## Q20. Design the Audit Fields

A team currently saves only `status = success` for every run. Suggest a better audit record.

**Suggested answer:** a run should record **what was processed, what happened to every row, and the business result**:

| Field | Why |
|---|---|
| `run_id` / `run_date` | Identifies which run and which day of data |
| `source_count` | How many rows arrived |
| `duplicate_count` | How many repeated order IDs were removed |
| `rejected_count` (and counts per `rejection_reason`) | How many rows failed validation, and why |
| `valid_count` | How many rows passed |
| `reconciles` | `source_count == valid_count + rejected_count` |
| `completed_order_count` | How many orders count as revenue |
| `completed_revenue_gbp` | The business total, to compare with the report |
| `status` | `success` only if every check passed |

`status = success` alone only says the code finished. It cannot show lost rows, unexpected rejections, or a wrong total.

In [16]:
# Example of a richer audit record (values from Q17)
better_audit = {
    "run_id": "run_20260801",
    "source_count": 6,
    "duplicate_count": 0,
    "rejected_count": 2,
    "rejected_by_reason": {"invalid_quantity": 1, "missing_price": 1},
    "valid_count": 4,
    "completed_order_count": 3,
    "completed_revenue_gbp": 120.0,
}
better_audit["reconciles"] = (
    better_audit["source_count"] == better_audit["valid_count"] + better_audit["rejected_count"]
)
better_audit["status"] = "success" if better_audit["reconciles"] else "failed"

better_audit

{'run_id': 'run_20260801',
 'source_count': 6,
 'duplicate_count': 0,
 'rejected_count': 2,
 'rejected_by_reason': {'invalid_quantity': 1, 'missing_price': 1},
 'valid_count': 4,
 'completed_order_count': 3,
 'completed_revenue_gbp': 120.0,
 'reconciles': True,
 'status': 'success'}

---
# Final Check

You should now be able to:

* validate customer and order data;
* decide when to fix or reject a value;
* validate a many to one join;
* create derived columns;
* preserve raw data;
* build a curated report;
* create audit records;
* run basic data quality checks;
* reconcile source, valid, and rejected counts.